# Network lab

A network is the simulator's map: named nodes, directed quantum/classical links, and route choices. The map does not move photons by itself. Delivery still goes through ports, connections, and the timeline.


In [ ]:
from dataclasses import dataclass, field
from math import log10

from simyuj.components import Port, PortDelivery, PortDirection, PortKind
from simyuj.engine.component import Component
from simyuj.engine.timeline import Timeline
from simyuj.metrics.path import route_success_probability, total_link_metric
from simyuj.network import Network, Node
from simyuj.network.planning import best_route_by_link_cost, candidate_routes, rank_routes
from simyuj.network.routing import RoutePlanner
from simyuj.network.topology import NetworkTopology


## 1. Build a small metro map

We will use a small metro network: Alice, Bob, and two relay sites.


In [ ]:
network = Network("metro_lab")

nodes = {
    node_id: Node(node_id)
    for node_id in ("alice", "north_relay", "south_relay", "bob")
}

for node in nodes.values():
    network.add_node(node)

print("Nodes in the network:", tuple(network.nodes))


## 2. Nodes name local devices and ports

Nodes are local namespaces. Devices own ports; a node only gives them local names.


In [ ]:
ACTION_RECEIVE_CLASSICAL = "receive_classical"


@dataclass(slots=True)
class ControlSender(Component):
    device_id: str
    output_port: Port = field(init=False)

    def __post_init__(self):
        self.output_port = Port(
            name="out",
            owner=self,
            owner_id=self.device_id,
            port_kind=PortKind.CLASSICAL,
            direction=PortDirection.EGRESS,
        )

    def handle_event(self, event, timeline):
        raise RuntimeError(f"{self.device_id} was not expecting an event")


In [ ]:
@dataclass(slots=True)
class ControlReceiver(Component):
    device_id: str
    received: list[tuple[int, PortDelivery]] = field(default_factory=list)
    input_port: Port = field(init=False)

    def __post_init__(self):
        self.input_port = Port(
            name="in",
            owner=self,
            owner_id=self.device_id,
            port_kind=PortKind.CLASSICAL,
            direction=PortDirection.INGRESS,
        )

    def handle_event(self, event, timeline):
        if event.action != ACTION_RECEIVE_CLASSICAL:
            raise RuntimeError(f"unexpected action: {event.action}")
        self.received.append((timeline.current_time, event.payload_ref))


In [ ]:
alice_sender = ControlSender("alice_control_sender")
bob_receiver = ControlReceiver("bob_control_receiver")

nodes["alice"].add_device("control_sender", alice_sender)
nodes["alice"].register_port("control_out", alice_sender.output_port)

nodes["bob"].add_device("control_receiver", bob_receiver)
nodes["bob"].register_port("control_in", bob_receiver.input_port)

print("Alice devices:", tuple(nodes["alice"].devices))
print("Alice port aliases:", tuple(nodes["alice"].ports))
print("Bob devices:", tuple(nodes["bob"].devices))
print("Bob port aliases:", tuple(nodes["bob"].ports))


## 3. Add physical links with real numbers

A link is directed graph reachability. We keep real-world numbers beside the links and use them for route scoring.


In [ ]:
@dataclass(frozen=True, slots=True)
class FiberSpan:
    length_km: float
    loss_db_per_km: float
    connector_loss_db: float

    @property
    def total_loss_db(self) -> float:
        return self.length_km * self.loss_db_per_km + self.connector_loss_db

    @property
    def photon_survival(self) -> float:
        return 10 ** (-self.total_loss_db / 10)


In [ ]:
quantum_spans = {
    "q_alice_bob_direct": ("alice", "bob", FiberSpan(88.0, 0.20, 1.1)),
    "q_alice_north": ("alice", "north_relay", FiberSpan(40.0, 0.20, 0.7)),
    "q_north_bob": ("north_relay", "bob", FiberSpan(42.0, 0.20, 0.8)),
    "q_alice_south": ("alice", "south_relay", FiberSpan(35.0, 0.22, 1.2)),
    "q_south_bob": ("south_relay", "bob", FiberSpan(47.0, 0.22, 1.4)),
}

for link_id, (source, target, span) in quantum_spans.items():
    network.add_quantum_link(link_id, source, target, channel=span)

print("Quantum link count:", len(network.quantum_links))
for link_id, link in network.quantum_links.items():
    span = link.transport
    print(f"{link_id}: {link.source_node_id} -> {link.target_node_id}, {span.length_km:.1f} km, {span.total_loss_db:.2f} dB")


In [ ]:
classical_spans = {
    "c_alice_bob": ("alice", "bob", 88.0),
    "c_bob_alice": ("bob", "alice", 88.0),
}

for link_id, (source, target, length_km) in classical_spans.items():
    network.add_classical_link(link_id, source, target, channel={"length_km": length_km})

print("Classical links:", tuple(network.classical_links))
print("All topology edges in deterministic order:")
for edge in network.edges:
    print(f"  {edge.link_id}: {edge.source_node_id} -> {edge.target_node_id} ({edge.port_kind.name.lower()})")


## 4. Ask topology questions

Topology answers graph questions. It ignores runtime wires.


In [ ]:
topology = NetworkTopology(network)

print("Topology nodes:", topology.nodes())
print("Alice quantum neighbors:", network.neighbors("alice", port_kind=PortKind.QUANTUM))
print("Alice classical neighbors:", network.neighbors("alice", port_kind=PortKind.CLASSICAL))
print("Bob quantum neighbors:", network.neighbors("bob", port_kind=PortKind.QUANTUM))
print("Bob classical neighbors:", network.neighbors("bob", port_kind=PortKind.CLASSICAL))


In [ ]:
print("Quantum edge alice -> bob:", network.has_edge("alice", "bob", port_kind=PortKind.QUANTUM))
print("Quantum edge bob -> alice:", network.has_edge("bob", "alice", port_kind=PortKind.QUANTUM))
print("Classical edge bob -> alice:", network.has_edge("bob", "alice", port_kind=PortKind.CLASSICAL))


## 5. Inspect routes as data

Routes are metadata: nodes, links, hop count, and port kinds.


In [ ]:
def show_route(label, route):
    print(label)
    if route is None:
        print("  no route")
        return
    print("  nodes:", " -> ".join(route.node_ids))
    print("  links:", route.link_ids)
    print("  hops:", route.hops)
    print("  kinds:", tuple(kind.name.lower() for kind in route.port_kinds))


In [ ]:
fewest_quantum = network.fewest_hops_path(
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
)

show_route("Fewest-hop quantum route", fewest_quantum)


In [ ]:
def span_for(link_id):
    return network.get_link(link_id).transport


def route_distance_km(route):
    return total_link_metric(
        route,
        {link_id: span.length_km for link_id, (_, _, span) in quantum_spans.items()},
        field_name="distance_km",
    )


shortest_fiber = network.lowest_cost_path(
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    link_cost=lambda link: link.transport.length_km,
)

show_route("Lowest-distance quantum route", shortest_fiber)
print("Fewest-hop distance:", route_distance_km(fewest_quantum), "km")
print("Lowest-distance total:", route_distance_km(shortest_fiber), "km")


## 6. Compare path choices

Fewest hops and best physical path can disagree. That is why route costs are supplied by you.


In [ ]:
planner = RoutePlanner(topology)

quantum_candidates = candidate_routes(
    planner,
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    max_hops=2,
)

print("Candidate quantum routes with at most two hops:")
for route in quantum_candidates:
    print(" ", route.link_ids, "via", " -> ".join(route.node_ids))


In [ ]:
loss_db_by_link = {
    link_id: span.total_loss_db
    for link_id, (_, _, span) in quantum_spans.items()
}

survival_by_link = {
    link_id: span.photon_survival
    for link_id, (_, _, span) in quantum_spans.items()
}

print("Per-link quantum numbers:")
for link_id in sorted(quantum_spans):
    print(
        f"{link_id}: "
        f"{quantum_spans[link_id][2].length_km:5.1f} km, "
        f"loss {loss_db_by_link[link_id]:5.2f} dB, "
        f"single-pass survival {survival_by_link[link_id] * 100:6.3f}%"
    )


In [ ]:
print("Candidate route table:")
for route in quantum_candidates:
    loss_db = total_link_metric(route, loss_db_by_link, field_name="loss_db")
    survival = route_success_probability(route, survival_by_link)
    print(
        f"{route.link_ids}: "
        f"{route_distance_km(route):5.1f} km, "
        f"{loss_db:5.2f} dB, "
        f"route survival {survival * 100:6.3f}%"
    )


In [ ]:
ranked_by_distance = rank_routes(quantum_candidates, route_distance_km)

print("Ranked by fiber length:")
for item in ranked_by_distance:
    print(f"score {item.score:5.1f} km -> {item.route.link_ids}")


In [ ]:
ranked_by_loss = rank_routes(
    quantum_candidates,
    lambda route: total_link_metric(route, loss_db_by_link, field_name="loss_db"),
)

print("Ranked by optical loss:")
for item in ranked_by_loss:
    print(f"score {item.score:5.2f} dB -> {item.route.link_ids}")


In [ ]:
ranked_by_survival = rank_routes(
    quantum_candidates,
    lambda route: 1.0 - route_success_probability(route, survival_by_link),
)

print("Ranked by photon survival:")
for item in ranked_by_survival:
    survival = 1.0 - item.score
    print(f"survival {survival * 100:6.3f}% -> {item.route.link_ids}")


In [ ]:
best_by_loss = best_route_by_link_cost(quantum_candidates, loss_db_by_link)

show_route("Best route when loss is the cost", best_by_loss)


## 7. Separate classical and quantum paths

Classical and quantum routes are separate searches.


In [ ]:
classical_to_bob = network.fewest_hops_path(
    "alice",
    "bob",
    port_kind=PortKind.CLASSICAL,
)

classical_to_alice = network.fewest_hops_path(
    "bob",
    "alice",
    port_kind=PortKind.CLASSICAL,
)

quantum_back_to_alice = network.fewest_hops_path(
    "bob",
    "alice",
    port_kind=PortKind.QUANTUM,
)

show_route("Classical Alice -> Bob", classical_to_bob)
show_route("Classical Bob -> Alice", classical_to_alice)
show_route("Quantum Bob -> Alice", quantum_back_to_alice)


## 8. Cross into runtime wires

Now cross the boundary: a runtime wire schedules delivery. It still does not create a topology edge.


In [ ]:
timeline = Timeline(master_seed=7)

control_wire = network.wire_ports(
    "control_wire_alice_bob",
    nodes["alice"].get_port("control_out"),
    nodes["bob"].get_port("control_in"),
    target_action=ACTION_RECEIVE_CLASSICAL,
)

print("Runtime wires:", tuple(network.wires))
print("Topology edge count after wiring:", len(network.edges))


In [ ]:
light_in_fiber_km_per_us = 0.2
classical_length_km = network.get_link("c_alice_bob").transport["length_km"]
arrival_tick = round(classical_length_km / light_in_fiber_km_per_us)

payload = {
    "message": "route selected",
    "quantum_links": best_by_loss.link_ids,
    "estimated_loss_db": total_link_metric(best_by_loss, loss_db_by_link, field_name="loss_db"),
}

event = control_wire.transmit(
    payload,
    timeline,
    time=arrival_tick,
    source=alice_sender,
)

print("Scheduled event id:", event.event_id)
print("Scheduled delivery tick:", arrival_tick)
print("Bob has received before the run:", len(bob_receiver.received))


In [ ]:
timeline.run_until(arrival_tick)

print("Bob has received after the run:", len(bob_receiver.received))
time, delivery = bob_receiver.received[0]
print("Delivery time:", time)
print("Connection id:", delivery.connection_id)
print("Payload:", delivery.payload)


## 9. Change the map and rerun the route

One last experiment: add a new physical span and rerun the route cells.


In [ ]:
network.add_quantum_link(
    "q_alice_bob_new_fiber",
    "alice",
    "bob",
    channel=FiberSpan(76.0, 0.18, 0.6),
)

quantum_spans["q_alice_bob_new_fiber"] = (
    "alice",
    "bob",
    network.get_link("q_alice_bob_new_fiber").transport,
)
loss_db_by_link["q_alice_bob_new_fiber"] = network.get_link("q_alice_bob_new_fiber").transport.total_loss_db
survival_by_link["q_alice_bob_new_fiber"] = network.get_link("q_alice_bob_new_fiber").transport.photon_survival

print("New Alice quantum neighbors:", network.neighbors("alice", port_kind=PortKind.QUANTUM))
print("New quantum links from Alice:")
for edge in network.outgoing_edges("alice", port_kind=PortKind.QUANTUM):
    span = network.get_link(edge.link_id).transport
    print(f"  {edge.link_id}: {span.length_km:.1f} km, {span.total_loss_db:.2f} dB")


In [ ]:
refreshed_candidates = candidate_routes(
    planner,
    "alice",
    "bob",
    port_kind=PortKind.QUANTUM,
    max_hops=2,
)

print("Routes after adding the new span:")
for route in refreshed_candidates:
    loss_db = total_link_metric(route, loss_db_by_link, field_name="loss_db")
    print(f"{route.link_ids}: {route_distance_km(route):5.1f} km, {loss_db:5.2f} dB")


In [ ]:
refreshed_best = best_route_by_link_cost(refreshed_candidates, loss_db_by_link)
show_route("Best route after the new span", refreshed_best)


## Keep this model in your head

The useful habit: keep the network as the map, keep physical numbers as route metrics, and use ports plus the timeline when something must actually move.
